<a href="https://colab.research.google.com/github/Daksh-Mistry/Depression-Analysis/blob/main/DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text

In [ ]:
!pip -q install pandas
!pip -q install numpy

In [ ]:
!wget 'https://dcapswoz.ict.usc.edu/wwwdaicwoz/train_split_Depression_AVEC2017.csv'

In [ ]:
!wget -q 'https://dcapswoz.ict.usc.edu/wwwdaicwoz/dev_split_Depression_AVEC2017.csv'

## See Data

Single Sample

In [ ]:
!wget -q https://dcapswoz.ict.usc.edu/wwwdaicwoz/300_P.zip
!unzip -o -q 300_P.zip

In [ ]:
import pandas as pd
df = pd.read_csv('/content/300_TRANSCRIPT.csv', sep='\t')
df.info()
df.describe()
df['speaker'].value_counts()

In [ ]:
!rm -rf 300_*

Lable DataSet

In [ ]:
Lable = pd.read_csv('/content/train_split_Depression_AVEC2017.csv')
Lable.head()

In [ ]:
test_lable = pd.read_csv('/content/dev_split_Depression_AVEC2017.csv')

In [ ]:
Lable.info()
Lable.describe()
Lable['PHQ8_Binary'].value_counts()

## New DataFrame

In [ ]:
import pandas as pd
Lable = pd.read_csv('/content/train_split_Depression_AVEC2017.csv')
DataList = []
for i in Lable['Participant_ID']:
  url = f'https://dcapswoz.ict.usc.edu/wwwdaicwoz/{i}_P.zip'
  !wget -q $url
  !unzip -o -q {i}_P.zip
  df = pd.read_csv(f'{i}_TRANSCRIPT.csv', sep='\t')
  lines = df.loc[df['speaker'] == 'Participant','value'].dropna()
  string = " ".join(lines.astype(str))
  # lable = Lable.loc(Lable['Participant_ID'] == i, 'PHQ8_Binary')
  label = Lable.loc[Lable['Participant_ID'] == i, 'PHQ8_Binary'].values[0]
  DataList.append({
      'Text' : string,
      'PHQ8_Binary' : label
  })
  !rm -rf {i}_*

In [ ]:
import pandas as pd
test_lable = pd.read_csv('/content/dev_split_Depression_AVEC2017.csv')
test_data = []
for i in test_lable['Participant_ID']:
  url = f'https://dcapswoz.ict.usc.edu/wwwdaicwoz/{i}_P.zip'
  !wget -q $url
  !unzip -o -q {i}_P.zip
  df = pd.read_csv(f'{i}_TRANSCRIPT.csv', sep='\t')
  lines = df.loc[df['speaker'] == 'Participant','value'].dropna()
  string = " ".join(lines.astype(str))
  # lable = Lable.loc(Lable['Participant_ID'] == i, 'PHQ8_Binary')
  label = test_lable.loc[test_lable['Participant_ID'] == i, 'PHQ8_Binary'].values[0]
  test_data.append({
      'Text' : string,
      'PHQ8_Binary' : label
  })
  !rm -rf {i}_*

In [ ]:
import pandas as pd

# Convert DataList to a DataFrame
df_final = pd.DataFrame(DataList)

# Display the first few rows of the new DataFrame
display(df_final.head())

In [ ]:
# Save the DataFrame to a CSV file
df_final.to_csv('processed_data.csv', index=False)

print('DataList successfully saved to processed_data.csv')

In [ ]:
import pandas as pd

In [ ]:
df_test = pd.DataFrame(test_data)
df_test.to_csv('test_data.csv',index=False)

You can load the data back into a DataFrame at any time using the following code:

In [ ]:
# To load the data back:
df = pd.read_csv('processed_data.csv')
display(df.head())

In [ ]:
df_test = pd.read_csv('test_data.csv')
display(df.head())

Encoding Process Using Tf-Idf

In [ ]:
!pip -q install scikit-learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
Encoder = TfidfVectorizer(stop_words='english',min_df=2,ngram_range=(1,2),max_features=1000)
train_text = Encoder.fit_transform(df['Text'])
train_lable = df['PHQ8_Binary']
test_text = Encoder.transform(df_test['Text'])
test_lable = df_test['PHQ8_Binary']

In [ ]:
train_text.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
model1 = LogisticRegression(class_weight='balanced')
model1.fit(train_text,train_lable)

In [ ]:
y = model.predict(test_text)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Print the coefficients and intercept of the logistic regression model
print("Coefficient:", model.coef_[0][0])
print("Intercept:", model.intercept_[0])

# Calculate and print accuracy
accuracy = accuracy_score(test_lable, y)
print(f"Accuracy: {accuracy:.4f}")

# Print classification report (includes precision, recall, f1-score, support)
print("\nClassification Report:")
print(classification_report(test_lable, y))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(test_lable, y))

Prediction

In [ ]:
probs = model.predict_proba(test_text)

In [ ]:
print(probs[:2])

In [ ]:
print(model.classes_)

In [ ]:
# Custom threshold (binary)
threshold = 0.45
y_changed = (probs[:, 1] >= threshold).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(test_lable, y_changed)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(test_lable, y_changed))

print("\nConfusion Matrix:")
print(confusion_matrix(test_lable, y_changed))

## Bert Version

In [ ]:
!pip -q install torch numpy pandas scikit-learn transformers

In [ ]:
import pandas as pd
df_test = pd.read_csv('test_data.csv')
df = pd.read_csv('processed_data.csv')

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('distilbert-base-uncased')
tokenized_df = tokenizer(df['Text'].tolist(), padding=True, truncation=True, return_tensors='pt')
tokenized_df_test = tokenizer(df_test['Text'].tolist(), padding=True, truncation=True, return_tensors='pt')

In [ ]:
tokenized_df_test

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(tokenized_df['input_ids'],
                        tokenized_df['attention_mask'],
                        torch.tensor(df['PHQ8_Binary'].values))
test_dataset = TensorDataset(tokenized_df_test['input_ids'],
                             tokenized_df_test['attention_mask'],
                             torch.tensor(df_test['PHQ8_Binary'].values))
train_DataLoader = DataLoader(dataset , batch_size = 32, shuffle=True)
test_DataLoader = DataLoader(test_dataset, batch_size = 32 , shuffle=True)

In [ ]:
train_DataLoader

In [ ]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
device

In [ ]:
import torch
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
import numpy as np

# -------------------------------
# 1. Compute class weights from training labels
# -------------------------------
# Assuming train_labels is a list/numpy array of your training labels (0/1)
train_labels = df['PHQ8_Binary'].values   # or wherever you have them
class_counts = np.bincount(train_labels)   # e.g., [900, 100]
total = len(train_labels)
# Inverse frequency weighting
weights = total / (len(class_counts) * class_counts)   # array of length num_classes
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

# -------------------------------
# 2. Create weighted loss function
# -------------------------------
loss_fn = CrossEntropyLoss(weight=class_weights)

# -------------------------------
# 3. Training loop with custom loss
# -------------------------------
optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(3):
    model.train()
    for batch in train_DataLoader:
        input_ids, attention_mask, batch_labels = [b.to(device) for b in batch]

        # Forward pass WITHOUT labels (so model only returns logits)
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Compute weighted loss
        loss = loss_fn(logits, batch_labels)

        # Backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    # -------------------------------
    # 4. Validation (unchanged, just for monitoring)
    # -------------------------------
    model.eval()
    # total_val_loss = 0
    # correct = 0
    # total = 0
    # with torch.no_grad():
    #     for batch in val_loader:
    #         input_ids, attention_mask, batch_labels = [b.to(device) for b in batch]
    #         outputs = model(input_ids, attention_mask=attention_mask)
    #         logits = outputs.logits
    #         # Optionally compute validation loss with the same loss_fn
    #         val_loss = loss_fn(logits, batch_labels)
    #         total_val_loss += val_loss.item()

    #         # Accuracy
    #         preds = torch.argmax(logits, dim=-1)
    #         correct += (preds == batch_labels).sum().item()
    #         total += batch_labels.size(0)

    # avg_val_loss = total_val_loss / len(val_loader)
    # val_acc = correct / total
    print(f"Epoch {epoch+1} completed")

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Make sure model is on the right device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Set model to evaluation mode
model.eval()

# Store predictions and true labels
all_preds = []
all_labels = []

# Disable gradient computation for efficiency
with torch.no_grad():
    for batch in test_DataLoader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Get predicted class (index with highest logit)
        preds = torch.argmax(logits, dim=-1)

        # Move to CPU and store
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# Audio

## Data Obervation

In [ ]:
!wget -q https://dcapswoz.ict.usc.edu/wwwdaicwoz/300_P.zip
!unzip -o -q 300_P.zip
!rm -rf 300_P.zip

In [ ]:
!pip -q install pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
df = pd.read_csv('/content/300_COVAREP.csv')
df.shape

In [ ]:
df.describe()

In [ ]:
import pandas as pd
df.iloc[:10,10:37]

In [ ]:
df.skew()

In [ ]:
df.kurtosis()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df.hist(figsize=(20, 15), bins=30, edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
df = pd.read_csv('/content/300_FORMANT.csv')
df.shape
df.info()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df.hist(figsize=(20, 15), bins=30, edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
df.describe()

In [ ]:
df.head(10)

## Data Prepration

I understood that audio data is direct numaric data so we dont have to modify that into anything, we just need to covert that into statistics and work it that, it might have too many features but that's how it works

In [ ]:
!pip -q install pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
!wget -q "https://dcapswoz.ict.usc.edu/wwwdaicwoz/dev_split_Depression_AVEC2017.csv"

In [ ]:
labels = pd.read_csv('/content/train_split_Depression_AVEC2017.csv')
labels.head()

In [ ]:
dev = pd.read_csv('/content/dev_split_Depression_AVEC2017.csv')
dev.head()

In [ ]:
def stats(DataFrame):
  matrix = DataFrame.values
  max = np.nanmax(matrix ,  axis=0)
  min = np.nanmin(matrix, axis=0)
  mean = np.nanmean(matrix, axis=0)
  median = np.nanmedian(matrix, axis=0)
  std = np.nanstd(matrix, axis=0)
  var = np.nanvar(matrix, axis=0)
  return np.concatenate([max,min,mean,median,std,var])

In [ ]:
data_list = []
for num in labels["Participant_ID"]:
  url = f"https://dcapswoz.ict.usc.edu/wwwdaicwoz/{num}_P.zip"
  !wget -q $url
  !unzip -o -q {num}_P.zip
  data1 = pd.read_csv(f'/content/{num}_FORMANT.csv',header=None)
  data2 = pd.read_csv(f'/content/{num}_COVAREP.csv',header=None)
  combined = pd.concat([data1, data2], axis=1)   # shape (n_frames, 78)
  feature_vector = stats(combined)               # aggregated vector

  # Get label
  label = labels.loc[labels['Participant_ID'] == num, 'PHQ8_Binary'].iloc[0]

  # Append to list
  data_list.append({
      "Data": feature_vector,
      "Label": label
  })
  !rm -rf {num}_*

In [ ]:
df = pd.DataFrame(data_list)
df.to_csv("audio_data.csv",index=False)

In [ ]:
data_list = []
for num in dev["Participant_ID"]:
  url = f"https://dcapswoz.ict.usc.edu/wwwdaicwoz/{num}_P.zip"
  !wget -q $url
  !unzip -o -q {num}_P.zip
  data1 = pd.read_csv(f'/content/{num}_FORMANT.csv',header=None)
  data2 = pd.read_csv(f'/content/{num}_COVAREP.csv',header=None)
  combined = pd.concat([data1, data2], axis=1)   # shape (n_frames, 78)
  feature_vector = stats(combined)               # aggregated vector

  # Get label
  label = dev.loc[dev['Participant_ID'] == num, 'PHQ8_Binary'].iloc[0]

  # Append to list
  data_list.append({
      "Data": feature_vector,
      "Label": label
  })
  !rm -rf {num}_*
df = pd.DataFrame(data_list)
df.to_csv("test_audio_data.csv",index=False)

In [ ]:
df = pd.DataFrame(data_list)
df.to_csv("test_audio_data.csv",index=False)

In [ ]:
train_audio = pd.read_csv("/content/audio_data.csv")
print(type(train_audio['Data'][0]))

In [ ]:
test_audio = pd.read_csv("/content/test_audio_data.csv")
print(type(train_audio.iloc[0]["Data"]))

In [ ]:
def convert(s):
  s = s.strip('[]')
  return np.array(list(map(float,s.split())))

In [ ]:
train_audio['Data'] = train_audio['Data'].apply(convert)

In [ ]:
test_audio['Data'] = test_audio['Data'].apply(convert)

Practical Way to store data is not CSV for large data sets

In [ ]:
train_audio.to_pickle("audio_data.pkl")
test_audio.to_pickle("test_audio_data.pkl")

In [ ]:
train_audio = pd.read_pickle("/content/audio_data.pkl")
print(type(train_audio['Data'][0]))

In [ ]:
test_audio = pd.read_pickle("/content/test_audio_data.pkl")
print(type(train_audio.iloc[0]["Data"]))

In [ ]:
train_audio.head()

In [ ]:
test_audio.head()

Applaying scaler

In [ ]:
train_matrix = np.array(train_audio['Data'].tolist())
test_matrix = np.array(test_audio['Data'].tolist())

In [ ]:
train_matrix = np.nan_to_num(train_matrix)
test_matrix = np.nan_to_num(test_matrix)

In [ ]:
print(np.isnan(train_matrix).sum())
print(np.isnan(test_matrix).sum())

In [ ]:
print(np.isinf(train_matrix).sum())
print(np.isinf(test_matrix).sum())

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit_transform(train_matrix)
scaler.transform(test_matrix)

In [ ]:
std = np.std(train_matrix, axis=0)

In [ ]:
mask = np.isfinite(std) & (std > 0)

train_matrix = train_matrix[:, mask]
test_matrix  = test_matrix[:, mask]

In [ ]:
print(train_matrix.shape)
print(test_matrix.shape)

In [ ]:
np.std(train_matrix, axis=1)

In [ ]:
train_lable = np.array(train_audio['Label'].to_list())
test_lable = np.array(test_audio['Label'].to_list())

In [ ]:
from sklearn.linear_model import LogisticRegression
model2 = LogisticRegression(class_weight = 'balanced', max_iter=5000)
model2.fit(train_matrix,train_lable)

In [ ]:
y = model.predict(test_matrix)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

model.fit(train_matrix, train_lable)
y = model.predict(test_matrix)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()
model.fit(train_matrix, train_lable)
y = model.predict(test_matrix)

In [ ]:
print(y)
print(test_lable)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Print the coefficients and intercept of the logistic regression model
# print("Coefficient:", model.coef_[0][0])
# print("Intercept:", model.intercept_[0])

# Calculate and print accuracy
accuracy = accuracy_score(test_lable, y)
print(f"Accuracy: {accuracy:.4f}")

# Print classification report (includes precision, recall, f1-score, support)
print("\nClassification Report:")
print(classification_report(test_lable, y))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(test_lable, y))

# Audio + Text Combination

In [ ]:
!pip install -q scipy

In [ ]:
import scipy.sparse as sp

x_train = sp.hstack((train_matrix, train_text))
x_test = sp.hstack((test_matrix, test_text))

y_train = train_lable
y_test = test_lable

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight = 'balanced', max_iter=5000)
model.fit(x_train,y_train)
y = model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Print the coefficients and intercept of the logistic regression model
# print("Coefficient:", model.coef_[0][0])
# print("Intercept:", model.intercept_[0])

# Calculate and print accuracy
accuracy = accuracy_score(y_test, y)
print(f"Accuracy: {accuracy:.4f}")

# Print classification report (includes precision, recall, f1-score, support)
print("\nClassification Report:")
print(classification_report(y_test, y))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y))

In [ ]:
model2.fit(train_matrix, y_train)
model1.fit(train_text, y_train)

prob_audio = model2.predict_proba(test_matrix)
prob_text = model1.predict_proba(test_text)

combined_probabilities = (prob_audio + prob_text) / 2

final_predictions = (combined_probabilities[:, 1] >= 0.5).astype(int)

print("Ensemble Confusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

print("\nEnsemble Classification Report:")
print(classification_report(y_test, final_predictions))

#Video